**`validate_delivery`**

Scores a delivered inventory against its out-of-band references and writes
the result into the delivery itself.

Two references, neither of them an input to the inventory:

- CHEER hand-labeled survey points, for per-class occupancy accuracy.
- Shovels building permits, for county-level agreement and full
  confusion matrices of the vote and each of its inputs.

Everything lands in an `accuracies/` folder beside the bundle's four
files, as tables plus figures.

Only aggregate numbers are written there. The row-level linkage carries
survey addresses, so it stays in the cache.

# Configure

In [ ]:
import argparse
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import openplaces as op
from openplaces.io.curator.validation import (
    read_confusion_matrix,
    validation_context,
    write_confusion_report,
)
from openplaces.io.delivery import delivery_accuracy_dir

# Every reference, class vocabulary and evidence column is declared in
# the curate recipe's `validation:` block (plus the untracked
# validation-references sidecar for licence-restricted tables); see
# io.curator.validation.ValidationContext.
RECIPE_ID = 'US_footprint-openplaces-2026'

In [ ]:
parser = argparse.ArgumentParser(
    description='Score a delivered inventory and write the result into the delivery.'
)
parser.add_argument('--recipe_id', default=RECIPE_ID)
parser.add_argument('--admin_ids', nargs='*', default=None)
# Default None resolves to the recipe's own delivery folder, so the
# accuracies travel with the bundle they describe.
parser.add_argument(
    # The survey and permit references are NC-specific, so the scores land
    # in the Carolina bundle's accuracies/ by default; the recipe ships two
    # regions and delivery_accuracy_dir raises without a selector.
    '--region',
    default='cheer-eastern-nc',
)
parser.add_argument('--out_dir', default=None)
parser.add_argument(
    '--min_scored',
    type=int,
    default=30,
    help='Counties with fewer scored rows are tabulated but left off the figure',
)
parser.add_argument(
    '--min_stratum_rows',
    type=int,
    default=10,
    help='Strata with fewer reference rows are not written to any matrix',
)
parser.add_argument('--verbose', action='store_true')

# Test arguments

In [ ]:
import os

# The orchestrated `validate` job executes this notebook headless and
# passes its arguments through this variable; interactively the test
# string applies.
ARGS_TEST = os.environ.get(
    'OPENPLACES_NOTEBOOK_ARGS', '--recipe_id US_footprint-openplaces-2026 --verbose '
)

args_list = [x for x in ARGS_TEST.split(' ') if x != '']

args = parser.parse_args(args_list)

args

# Validate the delivery

## Where the accuracies go

In [ ]:
out_dir = (
    Path(args.out_dir)
    if args.out_dir
    else delivery_accuracy_dir(args.recipe_id, region=args.region)
)
out_dir.mkdir(parents=True, exist_ok=True)

written = []


def publish(obj, name, **kwargs):
    """Write one table or figure into the delivery, and record it."""
    if isinstance(obj, mpl.figure.Figure):
        path = out_dir / f'{name}.png'
        obj.savefig(path, dpi=200, bbox_inches='tight')
    else:
        if obj is None or not len(obj):
            return None
        path = out_dir / f'{name}.csv'
        obj.to_csv(path, **kwargs)
    written.append(path.name)
    return path


print(f'accuracies -> {out_dir}')

# Which references score this region: the survey where the recipe's
# ground truth names it, and the permit reference whose declared region
# it is (none when the untracked sidecar is absent).
survey_ctx = validation_context(args.recipe_id)
CLASSES = list(survey_ctx.classes)
reference_regions = survey_ctx.reference_regions()
permit_state = next(
    (
        key
        for key, region in reference_regions.items()
        if key != 'ground_truth' and region == args.region
    ),
    None,
)
permit_ctx = validation_context(args.recipe_id, permit_state) if permit_state else None
print(
    f'survey: {reference_regions.get("ground_truth") == args.region}; '
    f'permit reference: {permit_state}'
)

## Chart style

One accent for the delivered vote, one neutral for the inputs it
arbitrates, and a single-hue ramp for magnitude.

The accent pair was checked with a colorblind-separation validator rather
than by eye: worst adjacent pair dE 20.3 under protanopia, 19.1 under
tritanopia, well clear of the 8.0 floor.

Every bar carries its value as text, so the figures survive being printed
in grayscale and never rest on color alone.

In [ ]:
# Accent = the delivered vote; neutral = the evidence it arbitrates.
# Deliberately not one hue per source: the question these figures answer
# is "how does the product compare with its inputs", which is a
# two-group contrast, not six identities.
VOTE_COLOR = '#9070C8'
INPUT_COLOR = '#B8B4C4'
GRID_COLOR = '#DDDBE3'
INK = '#2A2A32'
# Single hue, light to dark: the heatmap encodes magnitude, not identity.
MAGNITUDE_CMAP = 'Purples'

plt.rcParams.update(
    {
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.edgecolor': GRID_COLOR,
        'axes.labelcolor': INK,
        'text.color': INK,
        'xtick.color': INK,
        'ytick.color': INK,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'font.size': 9,
    }
)


def _bar_labels(ax, bars, fmt='{:.3f}', pad=0.006):
    """Label each bar at its end, so the figure reads without color."""
    for bar in bars:
        width = bar.get_width()
        if not np.isfinite(width):
            continue
        ax.text(
            width + pad,
            bar.get_y() + bar.get_height() / 2,
            fmt.format(width),
            va='center',
            ha='left',
            fontsize=8,
        )

## Occupancy against the CHEER hand labels

Per-class F1 for the delivered vote and for every input it arbitrates.

An input that outscores the vote on a class is evidence the vote is
discarding, which a single headline number cannot show.

In [ ]:
linked = None
scores = None
if reference_regions.get('ground_truth') == args.region:
    counties = (
        tuple(args.admin_ids) if args.admin_ids else survey_ctx.survey_admin_ids()
    )
    linked = survey_ctx.link_ground_truth(counties, verbose=args.verbose)
    # Also writes the survey confusion matrices and accuracies of every
    # source, pooled and per stratum.
    scores = survey_ctx.score_sources(linked, out_dir, min_rows=args.min_stratum_rows)
    written += [
        f'{args.recipe_id}_occupancy-survey{suffix}'
        for suffix in ('_confusion.csv', '_accuracy.csv', '_confusion.json')
    ]
    publish(scores, 'occupancy-accuracy-by-source', index=False)
    print(f'{len(linked):,} linked survey points')
else:
    print(f'the survey does not score {args.region}; skipped')
scores

In [ ]:
# Only where the survey scores this region.
if scores is not None:
    # One panel per class rather than grouped bars: the comparison the reader
    # makes is within a class, across sources, and grouping would interleave
    # the two.
    plot_classes = [*CLASSES, 'ALL']
    fig, axes = plt.subplots(
        1, len(plot_classes), figsize=(3.3 * len(plot_classes), 3.6), sharex=True
    )
    for ax, cls in zip(axes, plot_classes, strict=True):
        block = scores[scores['class'].eq(cls)].dropna(subset=['f1']).sort_values('f1')
        colors = [
            VOTE_COLOR if s == 'final_vote' else INPUT_COLOR for s in block['source']
        ]
        bars = ax.barh(block['source'], block['f1'], color=colors, height=0.62)
        _bar_labels(ax, bars)
        ax.set_title(cls, fontsize=10, pad=8)
        ax.set_xlim(0, 1.15)
        ax.xaxis.grid(True, color=GRID_COLOR, linewidth=0.6)
        ax.set_axisbelow(True)
        ax.tick_params(length=0)
    axes[0].set_xlabel('F1 vs hand labels')
    fig.suptitle(
        'Occupancy accuracy: the delivered vote (purple) against its inputs',
        y=1.04,
        fontsize=11,
    )
    publish(fig, 'occupancy-accuracy-by-source')
    plt.show()

## Occupancy against building permits

Permits are the second out-of-band reference. Coverage varies enormously
by county, so agreement is always reported next to the count it rests on.

Only the strong tiers are scored: permits reaching a footprint by parcel
id, or by address with a unanimous mode over at least two permits.

In [ ]:
rows = []
permit_frames = []
counties = permit_ctx.reference_admin_ids() if permit_ctx else []
if args.admin_ids:
    counties = [county for county in counties if county in args.admin_ids]
for county in counties:
    permits = permit_ctx.load_reference(county)
    footprints = op.get_entities(args.recipe_id, county, missing='ignore', geom=False)
    if permits is None or footprints is None or footprints.empty:
        continue
    joined = footprints.join(
        permits[
            [
                'occupancy_type_mode',
                'occupancy_type_mode_pct',
                'n_permits_with_occupancy_type',
                'matched_via',
            ]
        ],
        how='inner',
    )
    joined['vote'] = permit_ctx.collapse_bands(joined['occupancy_type'])
    joined['tier'] = permit_ctx.reference_tier(joined)
    # Permit evidence is parcel-level, so a shed inherits the house's
    # class; only primary structures can be scored against it.
    primary = joined[
        joined['priority_on_parcel'].astype(object).isin(['primary', 'unknown'])
    ]
    strong = primary[primary['tier'].isin(permit_ctx.reference_strong_tiers)]
    if strong.empty:
        continue
    answered = strong[strong['vote'].notna()]
    rows.append(
        {
            'county': county,
            'n_scored': len(answered),
            'agreement': round(
                answered['vote'].eq(answered['occupancy_type_mode']).mean(), 4
            )
            if len(answered)
            else None,
            'coverage': round(primary['tier'].ne('none').mean(), 4),
        }
    )
    # The vote and each input, scored on the same rows. Held in memory
    # only: the permit mode per footprint is restricted row-level data,
    # and only the aggregate matrices below are written.
    frame = pd.DataFrame(permit_ctx.entity_source_values(strong), index=strong.index)
    frame['__reference'] = strong['occupancy_type_mode'].astype(object)
    frame['__county'] = county
    permit_frames.append(frame)

permit_accuracy = pd.DataFrame(
    rows, columns=['county', 'n_scored', 'agreement', 'coverage']
)
permit_accuracy = permit_accuracy.sort_values('agreement', ascending=False)
publish(permit_accuracy, 'permit-accuracy-by-county', index=False)
permit_accuracy

In [ ]:
if permit_accuracy.empty:
    print('no permit reference scores this region')
else:
    # Sorting the full list by agreement puts counties with one or two scored
    # rows on top at a flat 1.000, which reads as the best county in the
    # delivery and is nothing of the kind. The table keeps every county; the
    # figure shows only those with enough rows to rank, and carries n in the
    # label so the reader never sees a rate without its denominator.
    panel = permit_accuracy[permit_accuracy['n_scored'] >= args.min_scored].sort_values(
        'agreement'
    )
    thin = len(permit_accuracy) - len(panel)

    y = np.arange(len(panel))
    fig, ax = plt.subplots(figsize=(8.0, max(3.0, 0.42 * len(panel))))
    ax.barh(
        y + 0.19,
        panel['agreement'],
        height=0.34,
        color=VOTE_COLOR,
        label='agreement with permit mode',
    )
    ax.barh(
        y - 0.19,
        panel['coverage'],
        height=0.34,
        color=INPUT_COLOR,
        label='share of footprints with permit evidence',
    )
    for i, row in enumerate(panel.itertuples()):
        ax.text(
            row.agreement + 0.008,
            i + 0.19,
            f'{row.agreement:.3f}',
            va='center',
            fontsize=8,
        )
        ax.text(
            row.coverage + 0.008,
            i - 0.19,
            f'{row.coverage:.1%}',
            va='center',
            fontsize=8,
            color='#5A5A66',
        )
    ax.set_yticks(
        y,
        [
            f'{c}  (n={n:,})'
            for c, n in zip(panel['county'], panel['n_scored'], strict=True)
        ],
    )
    ax.set_xlim(0, 1.15)
    ax.set_xlabel('share')
    ax.xaxis.grid(True, color=GRID_COLOR, linewidth=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
    # On the figure, not the axes: the panel grows with the county
    # count, so an axes-fraction offset that clears the x-label at 11
    # counties collides with it at 30. bbox_inches='tight' keeps it in
    # the saved image.
    fig.legend(
        *ax.get_legend_handles_labels(),
        frameon=False,
        fontsize=8,
        loc='upper center',
        bbox_to_anchor=(0.5, 0.0),
        ncol=2,
    )
    ax.set_title(
        'Permit agreement, and the coverage it rests on\n'
        f'{len(panel)} counties with at least {args.min_scored} scored rows'
        + (f'; {thin} thinner counties are in the table only' if thin else ''),
        fontsize=11,
        pad=10,
    )
    publish(fig, 'permit-accuracy-by-county')
    plt.show()

## Where the vote and its inputs disagree with the permits

Rows are what the permits say, columns the delivered class: the usual
orientation, so a row's shares are producer's accuracy and a column's
are consumer's accuracy.

The matrices cover the vote and every input it arbitrates, pooled and
per county, and keep every row:

- `Secondary` counts outbuildings (the recipe's secondary class);
- `Non-residential` counts any other asserted class, on either side.
  The delivery resolves occupancy far more finely than a permit record
  does, so a permit's `Institutional` against a delivered `Church`
  lands there rather than reading as an error;
- `No class` counts footprints a source said nothing about. NSI and
  FEMA keep a non-residential class as an assertion, so their
  `No class` means no record, not a class outside the vocabulary.

What is worth reading is disagreement *within* a shared class: a
delivered `Single-Family` the permits call `Multi-Family`, or the
`Manufactured Home` / `Single-Family` split.

In [ ]:
if not permit_frames:
    print('no permit rows scored for this region; no permit matrices')
else:
    scored = pd.concat(permit_frames, ignore_index=True)
    sources = [column for column in scored.columns if not column.startswith('__')]
    paths = write_confusion_report(
        scored['__reference'],
        {source: scored[source] for source in sources},
        CLASSES,
        out_dir,
        f'{args.recipe_id}_permit-occupancy',
        strata={'county': scored['__county']},
        reference='building permits, occupancy mode per parcel',
        tier=', '.join(permit_ctx.reference_strong_tiers),
        notes='Primary footprints only; permit evidence is parcel-level.',
        min_rows=args.min_stratum_rows,
        **permit_ctx.matrix_labels(),
    )
    written += [path.name for path in paths.values()]
    del scored

    table = read_confusion_matrix(paths['confusion'], 'final_vote')
    shares = table.div(table.sum(axis=1).replace(0, 1), axis=0)
    fig, ax = plt.subplots(
        figsize=(1.6 + 0.9 * table.shape[1], 1.4 + 0.5 * table.shape[0])
    )
    im = ax.imshow(
        shares.to_numpy(), cmap=MAGNITUDE_CMAP, vmin=0, vmax=1, aspect='auto'
    )
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=35, ha='right')
    ax.set_yticks(range(table.shape[0]), table.index)
    ax.tick_params(length=0)
    for r in range(table.shape[0]):
        for c in range(table.shape[1]):
            n = table.iat[r, c]
            # Ink flips on the dark end so the count stays legible.
            ax.text(
                c,
                r,
                f'{n:,}',
                ha='center',
                va='center',
                fontsize=7,
                color='white' if shares.iat[r, c] > 0.55 else INK,
            )
    fig.colorbar(im, ax=ax, shrink=0.7, label='share of the permit class')
    ax.set_xlabel('delivered occupancy')
    ax.set_ylabel('permit occupancy')
    ax.set_title(
        f'Permits vs the delivered class, {len(permit_accuracy)} counties',
        fontsize=11,
        pad=10,
    )
    publish(fig, 'permit-confusion')
    plt.show()

## Manifest

A short README so the folder explains itself to someone who receives the
bundle without this repository.

In [ ]:
n_survey_points = 0 if linked is None else len(linked)
lines = [
    f'# Accuracy reporting for {args.recipe_id}, {args.region}',
    '',
    'Scored against references that are not inputs to the inventory.',
    '',
    f'- CHEER hand-labeled survey points: {n_survey_points:,} points.',
    f'- Shovels building permits: {len(permit_accuracy)} counties with '
    'scorable evidence.',
    '',
    'Every `*_confusion.csv` is a long-form confusion matrix (rows are the',
    'reference, columns the prediction) for each source, pooled and per',
    "stratum; `*_accuracy.csv` holds producer's and consumer's accuracy.",
    '',
    'Permit coverage varies by county by two orders of magnitude, so read',
    'every agreement number next to the count beside it.',
    '',
    '## Files',
    '',
]
lines += [f'- `{name}`' for name in sorted(written)]
readme = out_dir / 'README.md'
readme.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(f'wrote {len(written)} files + README.md to {out_dir}')
for name in sorted(written):
    print(f'  {name}')

---
# Convert to script

*The above line and heading identify the end of the script.*

In [ ]:
from openplaces.flow import convert_to_script

try:
    convert_to_script(commit=True)
except Exception as error:
    # Headless execution (nbconvert) has no notebook context to resolve
    # the caller path from; run this cell interactively to commit the
    # script. Stripped from the converted script either way.
    print(f'convert_to_script skipped: {error}')

# Test script

In [ ]:
# from openplaces.flow import test_script

# test_script(*args_list, committed=True)

# Inspect results

In [ ]:
sorted(p.name for p in out_dir.iterdir())